In [1]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')
!unzip "/content/drive/My Drive/processed_data.csv.zip"

t0 = time.time()
df = pd.read_csv('processed_data.csv')
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(['DATE', 'permno']).reset_index(drop=True)

print(f"  Shape: {df.shape}  |  "
      f"{df['DATE'].min().date()} to {df['DATE'].max().date()}  |  "
      f"{df['permno'].nunique():,} stocks  |  loaded in {time.time()-t0:.1f}s")

TARGET = 'exret'

MACRO_COLS = ['tbl', 'd/p', 'e/p', 'b/m', 'tms', 'dfy', 'ntis', 'svar']

CHAR_COLS = [
    'mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom',
    'mom1m', 'mom6m', 'mom12m', 'mom36m', 'pricedelay', 'turn',
    'absacc', 'acc', 'age', 'agr', 'bm', 'bm_ia', 'cashdebt', 'cashpr',
    'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chinv', 'chpmia',
    'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep',
    'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'invest', 'lev', 'lgr',
    'mve_ia', 'operprof', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr',
    'pchgm_pchsale', 'pchquick', 'pchsale_pchinvt', 'pchsale_pchrect',
    'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'ps', 'quick', 'rd',
    'rd_mve', 'rd_sale', 'realestate', 'roic', 'salecash', 'saleinv',
    'salerec', 'secured', 'securedind', 'sgr', 'sin', 'sp', 'tang', 'tb',
    'aeavol', 'cash', 'chtx', 'cinvest', 'ear', 'nincr', 'roaq', 'roavol',
    'roeq', 'rsup', 'stdacc', 'stdcf', 'ms', 'baspread', 'ill', 'maxret',
    'retvol', 'std_dolvol', 'std_turn', 'zerotrade']

CHAR_COLS  = [c for c in CHAR_COLS  if c in df.columns]
MACRO_COLS = [c for c in MACRO_COLS if c in df.columns]
HAS_SIC2   = 'sic2' in df.columns

print(f"  Chars: {len(CHAR_COLS)}  |  Macros: {len(MACRO_COLS)}  |  SIC2: {HAS_SIC2}")

# feature matrix
t1 = time.time()

REQUIRED = CHAR_COLS + MACRO_COLS + [TARGET]
df = df.dropna(subset=REQUIRED).reset_index(drop=True)
print(f"  Clean rows: {len(df):,}")

chars  = df[CHAR_COLS].values.astype(np.float32)
macros = df[MACRO_COLS].values.astype(np.float32)

interactions = (chars[:, :, np.newaxis] * macros[:, np.newaxis, :])
interactions = interactions.reshape(len(df), len(CHAR_COLS) * len(MACRO_COLS))

X_base = np.hstack([chars, interactions])

if HAS_SIC2:
    sic2_dummies = pd.get_dummies(
        df['sic2'], prefix='sic2', drop_first=False
    ).values.astype(np.float32)
    X_all = np.hstack([X_base, sic2_dummies])
else:
    X_all = X_base

y_all   = df[TARGET].values.astype(np.float64)
years   = df['DATE'].dt.year.values
permnos = df['permno'].values
dates   = df['DATE'].values
mvel1   = df['mvel1'].values if 'mvel1' in df.columns else None

P = X_all.shape[1]
print(f"  Feature matrix: {X_all.shape}  ({X_all.nbytes/1e9:.2f} GB)  "
      f"built in {time.time()-t1:.1f}s")


# huber
def huber_weights(residuals, xi):
    abs_res = np.abs(residuals)
    w = np.where(abs_res <= xi, 1.0, xi / np.maximum(abs_res, 1e-10))
    return w

def fit_huber_irls(X, y, xi, theta_init=None, max_iter=30, tol=1e-5):
    N, P = X.shape

    # Initialise from warm start or plain OLS
    if theta_init is not None:
        theta = theta_init.copy()
    else:
        XtX = X.T @ X
        Xty = X.T @ y
        theta = np.linalg.solve(XtX + 1e-10 * np.eye(P), Xty)

    for _ in range(max_iter):
        residuals = y - X @ theta
        w = huber_weights(residuals, xi)

        Xw  = X * w[:, np.newaxis]
        XtWX = Xw.T @ X
        XtWy = Xw.T @ y

        theta_new = np.linalg.solve(XtWX + 1e-10 * np.eye(P), XtWy)

        if np.max(np.abs(theta_new - theta)) < tol:
            theta = theta_new
            break
        theta = theta_new

    return theta

# OOS R2
def oos_r2(y_true, y_pred):
    """Equation (19): R2 = 1 - SS_res / sum(r^2)  [zero-benchmark]"""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return 1.0 - np.sum((y_true - y_pred)**2) / np.sum(y_true**2)


# Expanding window
VALIDATION_END_YEAR = 1986
TEST_END_YEAR       = 2016

all_test_years = sorted(
    set(years[(years > VALIDATION_END_YEAR) & (years <= TEST_END_YEAR)])
)

print(f"\nRunning expanding-window OLS+H  "
      f"({all_test_years[0]}–{all_test_years[-1]})...")
t2 = time.time()

pred_dates   = []
pred_permnos = []
pred_y_true  = []
pred_y_pred  = []
pred_mvel1   = []

theta = None

for year in all_test_years:
    train_mask = years <= (year - 1)
    test_mask  = years == year

    X_train = X_all[train_mask].astype(np.float64)
    y_train = y_all[train_mask]
    X_test  = X_all[test_mask].astype(np.float64)
    y_test  = y_all[test_mask]

    # Huber threshold xi
    if theta is None:
        XtX   = X_train.T @ X_train
        Xty   = X_train.T @ y_train
        theta = np.linalg.solve(XtX + 1e-10 * np.eye(P), Xty)

    resid_init = y_train - X_train @ theta
    xi = np.percentile(np.abs(resid_init), 99.9)

    theta = fit_huber_irls(X_train, y_train, xi=xi, theta_init=theta)
    y_pred = X_test @ theta

    pred_dates.append(dates[test_mask])
    pred_permnos.append(permnos[test_mask])
    pred_y_true.append(y_test)
    pred_y_pred.append(y_pred)
    if mvel1 is not None:
        pred_mvel1.append(mvel1[test_mask])

    print(f"  {year}  |  train: {train_mask.sum():,}  |  "
          f"test: {test_mask.sum():,}  |  xi: {xi:.4f}  |  "
          f"elapsed: {time.time()-t2:.1f}s")
print(f"Loop finished in {time.time()-t2:.1f}s")

# Results
y_true_all = np.concatenate(pred_y_true)
y_pred_all = np.concatenate(pred_y_pred)
dates_all  = np.concatenate(pred_dates)
perm_all   = np.concatenate(pred_permnos)

r2_all = oos_r2(y_true_all, y_pred_all)

print(f"\n{'═'*65}")
print(f"  {'Subsample':<32}  {'OOS R2':>10}  {'Paper':>10}")
print(f"{'─'*65}")
print(f"  {'All stocks (panel)':<32}  {r2_all*100:>+10.4f}%  {'–3.46%':>10}")

if mvel1 is not None:
    mv_all     = np.concatenate(pred_mvel1)
    results_df = pd.DataFrame({
        'DATE': dates_all, 'mvel1': mv_all,
        'y_true': y_true_all, 'y_pred': y_pred_all
    })
    for label, largest, paper_val in [
        ('Top 1000 (largest)',     True,  -11.28),
        ('Bottom 1000 (smallest)', False,  -1.30),
    ]:
        fn  = (lambda g: g.nlargest(1000,  'mvel1')) if largest \
              else (lambda g: g.nsmallest(1000, 'mvel1'))
        sub = results_df.groupby('DATE', group_keys=False).apply(fn)
        r2  = oos_r2(sub['y_true'].values, sub['y_pred'].values)
        print(f"  {label:<32}  {r2*100:>+10.4f}%  {paper_val:>+10.2f}%")

print(f"{'═'*65}")
print(f"\nTotal wall time: {time.time()-t0:.1f}s")


out = pd.DataFrame({
    'DATE':   dates_all,
    'permno': perm_all,
    'y_true': y_true_all,
    'y_pred': y_pred_all,
})
out_path = 'ols_predictions.csv'
out.to_csv(out_path, index=False)
print(f"Predictions saved → {out_path}  ({len(out):,} rows)")

Mounted at /content/drive
Archive:  /content/drive/My Drive/processed_data.csv.zip
  inflating: processed_data.csv      
  inflating: __MACOSX/._processed_data.csv  
Loading data...
  Shape: (4096791, 111)  |  1957-01-31 to 2021-12-31  |  32,754 stocks  |  loaded in 88.3s
  Chars: 94  |  Macros: 8  |  SIC2: True

Building full feature matrix (once)...
  Clean rows: 4,096,791
  Feature matrix: (4096791, 920)  (15.08 GB)  built in 20.9s

Running expanding-window OLS+H  (1987–2016)...
  1987  |  train: 1,248,579  |  test: 82,945  |  xi: 1.0680  |  elapsed: 73.0s
  1988  |  train: 1,331,524  |  test: 84,185  |  xi: 1.0932  |  elapsed: 162.4s
  1989  |  train: 1,415,709  |  test: 81,865  |  xi: 1.1021  |  elapsed: 240.5s
  1990  |  train: 1,497,574  |  test: 80,783  |  xi: 1.1162  |  elapsed: 312.1s
  1991  |  train: 1,578,357  |  test: 79,789  |  xi: 1.1373  |  elapsed: 403.9s
  1992  |  train: 1,658,146  |  test: 81,614  |  xi: 1.2050  |  elapsed: 479.2s
  1993  |  train: 1,739,760  |  te

OSError: Cannot save file into a non-existent directory: '/mnt/user-data/outputs'